# Daily Offers — Revenue Performance

In [1]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

import pandas as pd
import plotly.express as px
#from google.cloud import bigquery
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html

In [2]:
refresh_data = False

In [3]:
query_location = './sql/daily_offers.sql'
query_name = 'daily_offers'

parameters = {
    'period_offset':365,
}

bqc = BigQueryConnector()
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 6.2 MB when run.
Estimated query cost: $0.00


In [4]:
data = pd.DataFrame()

if refresh_data:
    data = bqc.get(query=f'./sql/{query_name}.sql', is_path=True, query_parameters=parameters)
    data.to_pickle(f'./data/{query_name}.pkl')
else:
    data = pd.read_pickle(f'./data/{query_name}.pkl')

## Raw Data

In [5]:
data

,user_id,dt,product_id,product_type,product_theme,price_point_usd,product_price_group,first_seen,last_seen,n_trans,usd_revenue
0,DEAB88AAC54CAC35,2026-08-19,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,74.99,daily_offers_5099_plus,2026-08-19,2026-08-19,2,144.80
1,B46BE3571773CE42,2026-08-17,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,99.99,daily_offers_5099_plus,2026-08-17,2026-08-17,1,113.72
2,BE835972C0D41A71,2026-08-21,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,99.99,daily_offers_5099_plus,2026-08-21,2026-08-21,1,101.64
3,DEAB88AAC54CAC35,2026-08-20,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,49.99,daily_offers_2099_4999,2026-08-20,2026-08-20,2,101.57
4,6980B3A8474F4C40,2026-08-19,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,99.99,daily_offers_5099_plus,2026-08-19,2026-08-19,1,99.99
...,...,...,...,...,...,...,...,...,...,...,...
8574,106B0E0ED94D9F,2026-08-18,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-18,2026-08-18,1,0.42
8575,9CD41EBD47773AFE,2026-08-24,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-24,2026-08-24,1,0.42
8576,468E1C80CBE8CFB8,2026-08-30,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-30,2026-08-30,1,0.42
8577,D5BEB8FD4DBCD610,2026-08-31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-31,2026-08-31,1,0.42


## Product-Level Aggregation

`data` is now at user–day–product grain, so `first_seen`/`last_seen` in the raw rows are just each row's own `dt` — not useful on their own. Roll up to `product_summary` (one row per offer) to get real first/last-seen dates, `active_days`, and buyer counts; all the charts below aggregate from `data` or `product_summary` rather than assuming pre-aggregated rows.

`active_days` normalizes across offers that have been live for different lengths of time (e.g. new 2026Q3 offers vs. 2025Q4 offers that have run the full window) — `revenue_per_day` is the fairer basis for ranking offer performance than raw total revenue.

In [6]:
data['offer_quarter'] = data['product_id'].str.extract(r'(\d{4}Q\d)')
data['bundle_type'] = data['product_id'].str.extract(r'-[A-Za-z]+-([A-Za-z]+)(?:\s[\d.]+)?$')

product_summary = data.groupby(
    ['product_id', 'product_type', 'product_theme', 'price_point_usd', 'product_price_group',
     'offer_quarter', 'bundle_type'],
    as_index=False,
).agg(
    first_seen=('dt', 'min'),
    last_seen=('dt', 'max'),
    n_trans=('n_trans', 'sum'),
    n_buyers=('user_id', 'nunique'),
    usd_revenue=('usd_revenue', 'sum'),
)

product_summary['active_days'] = (product_summary['last_seen'] - product_summary['first_seen']).dt.days + 1
product_summary['revenue_per_day'] = (product_summary['usd_revenue'] / product_summary['active_days']).round(2)
product_summary['revenue_per_trans'] = (product_summary['usd_revenue'] / product_summary['n_trans']).round(2)
product_summary['revenue_share_pct'] = (product_summary['usd_revenue'] / product_summary['usd_revenue'].sum() * 100).round(2)

product_summary.sort_values('usd_revenue', ascending=False)

,product_id,product_type,product_theme,price_point_usd,product_price_group,offer_quarter,bundle_type,first_seen,last_seen,n_trans,n_buyers,usd_revenue,active_days,revenue_per_day,revenue_per_trans,revenue_share_pct
35,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,24.99,daily_offers_2099_4999,2025Q4,GardenBundle,2026-08-17,2026-08-31,556,316,14711.43,15,980.76,26.46,12.81
31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,9.99,daily_offers_0599_0999,2025Q4,GardenBundle,2026-08-17,2026-08-31,1246,716,13463.49,15,897.57,10.81,11.72
33,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,14.99,daily_offers_1099_1999,2025Q4,GardenBundle,2026-08-17,2026-08-31,589,424,9419.45,15,627.96,15.99,8.20
34,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,19.99,daily_offers_1099_1999,2025Q4,GardenBundle,2026-08-17,2026-08-31,441,317,9399.22,15,626.61,21.31,8.18
32,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,12.99,daily_offers_1099_1999,2025Q4,GardenBundle,2026-08-17,2026-08-31,630,458,8912.07,15,594.14,14.15,7.76
36,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,34.99,daily_offers_2099_4999,2025Q4,PartyBundle,2026-08-17,2026-08-31,220,128,7999.31,15,533.29,36.36,6.96
47,TimedAlbum-DailyOffers-2026Q3-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,14.99,daily_offers_1099_1999,2026Q3,GardenBundle,2026-08-26,2026-08-29,451,403,7338.80,4,1834.70,16.27,6.39
37,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,49.99,daily_offers_2099_4999,2025Q4,PartyBundle,2026-08-17,2026-08-31,116,52,5993.87,15,399.59,51.67,5.22
27,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,7.99,daily_offers_0599_0999,2025Q4,CafeBundle,2026-08-17,2026-08-31,518,433,4484.85,15,298.99,8.66,3.90
48,TimedAlbum-DailyOffers-2026Q3-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,24.99,daily_offers_2099_4999,2026Q3,GardenBundle,2026-08-26,2026-08-28,147,136,3865.08,3,1288.36,26.29,3.37


## Top Offers by Revenue

In [7]:
top_n = 15
top_offers = product_summary.nlargest(top_n, 'usd_revenue').sort_values('usd_revenue')
top_offers = top_offers.assign(
    label=top_offers['bundle_type'] + ' · ' + top_offers['offer_quarter'] + ' · $' + top_offers['price_point_usd'].astype(str)
)

fig = px.bar(
    top_offers, x='usd_revenue', y='label', orientation='h',
    custom_data=['n_trans', 'n_buyers', 'price_point_usd', 'revenue_share_pct'],
)
fig.update_traces(
    marker_color='steelblue',
    hovertemplate=(
        'Revenue: $%{x:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Buyers: %{customdata[1]:,.0f}<br>'
        'Price: $%{customdata[2]:.2f}<br>Share of total: %{customdata[3]:.1f}%<extra></extra>'
    ),
)
fig.update_layout(
    title=f'Top {top_n} Daily Offers by Total Revenue',
    xaxis=dict(title='Revenue (USD)', tickprefix='$'),
    yaxis=dict(title=None, automargin=True),
    height=550, width=1100,
    margin=dict(l=10),
)
fig.show()

## Revenue by Product Type

In [8]:
# Fixed identity color per product_type, kept stable regardless of ranking
type_colors = {
    'RotatingIncPack': '#2a78d6',
    'Rotating': '#eb6834',
    'NonPayerIncPack': '#1baf7a',
    'NonPayer': '#eda100',
}

by_type = product_summary.groupby('product_type', as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('usd_revenue', ascending=False)

fig = px.bar(
    by_type, x='product_type', y='usd_revenue', color='product_type',
    color_discrete_map=type_colors,
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    hovertemplate='Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Product Type',
    xaxis=dict(title=None),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=800,
    showlegend=False,
)
fig.show()

## Revenue by Bundle Type

In [29]:
# Fixed identity color per bundle_type, kept stable regardless of ranking
bundle_colors = {
    'CafeBundle': '#2a78d6',
    'GardenBundle': '#eb6834',
    'PartyBundle': '#1baf7a',
    'BundleSmall': '#eda100',
    'BundleMedium': '#e87ba4',
    'BundleLarge': '#008300',
}

by_bundle = product_summary.groupby(['bundle_type','offer_quarter'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('usd_revenue', ascending=False)

fig = px.bar(
    by_bundle, x='bundle_type', y='usd_revenue', color='offer_quarter',
    color_discrete_map=bundle_colors,
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    hovertemplate='Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Bundle Type',
    xaxis=dict(title=None),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=800,
    showlegend=False,
)
fig.show()

## Revenue by Price Point

In [34]:
by_price = product_summary.groupby(['price_point_usd','offer_quarter'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('price_point_usd')
by_price = by_price.assign(price_label=by_price['price_point_usd'].astype(str))

fig = px.bar(
    by_price, 
    x='price_label', 
    y='usd_revenue',
    color='offer_quarter',
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    #marker_color='steelblue',
    hovertemplate='Price: $%{x}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Price Point',
    xaxis=dict(title='Price (USD)', type='category'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=1000,
)
fig.show()

## Revenue by Offer Quarter

Raw totals — 2026Q3 offers have only been live a few days vs. the full window for 2025Q4, so this isn't a fair per-offer comparison (see the normalized ranking below).

In [11]:
by_quarter = product_summary.groupby('offer_quarter', as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    n_offers=('product_id', 'nunique'),
).sort_values('offer_quarter')

fig = px.bar(
    by_quarter, x='offer_quarter', y='usd_revenue',
    custom_data=['n_trans', 'n_offers'],
)
fig.update_traces(
    marker_color='steelblue',
    hovertemplate='Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offers: %{customdata[1]}<extra></extra>'
)
fig.update_layout(
    title='Revenue by Offer Quarter',
    xaxis=dict(title=None, type='category'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=800,
)
fig.show()

In [12]:
daily_by_quarter = data.groupby(['dt', 'offer_quarter'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_usd_revenue=('usd_revenue', 'mean'),
    avg_price_point_usd=('price_point_usd', 'mean')
).sort_values('dt')

quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

# Daily revenue per transaction by day and quarter

fig = px.bar(
    daily_by_quarter, x='dt', y='usd_revenue', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Revenue by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

# Daily transactions by day and quarter

fig = px.bar(
    daily_by_quarter, x='dt', y='n_trans', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Transactions: %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Transactions by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Transactions', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

# Average revenue per transaction by day and quarter

fig = px.line(
    daily_by_quarter, x='dt', y='avg_usd_revenue', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Average Revenue: $%{y:,.2f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Average Revenue by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Average Revenue (USD)', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()



# Average price_point  by day and quarter

fig = px.line(
    daily_by_quarter, x='dt', y='avg_price_point_usd', color='offer_quarter',
    color_discrete_map=quarter_colors,
    custom_data=['n_trans'],
)
fig.update_traces(
    mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Average Price Point: $%{y:,.2f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>',
)
fig.update_layout(
    title='Daily Average Price Point by Offer Quarter',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Average Price Point (USD)', tickprefix='$'),
    height=450, width=1100,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()



In [13]:
data

,user_id,dt,product_id,product_type,product_theme,price_point_usd,product_price_group,first_seen,last_seen,n_trans,usd_revenue,offer_quarter,bundle_type
0,DEAB88AAC54CAC35,2026-08-19,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,74.99,daily_offers_5099_plus,2026-08-19,2026-08-19,2,144.80,2025Q4,PartyBundle
1,B46BE3571773CE42,2026-08-17,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,99.99,daily_offers_5099_plus,2026-08-17,2026-08-17,1,113.72,2025Q4,PartyBundle
2,BE835972C0D41A71,2026-08-21,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,99.99,daily_offers_5099_plus,2026-08-21,2026-08-21,1,101.64,2025Q4,PartyBundle
3,DEAB88AAC54CAC35,2026-08-20,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,49.99,daily_offers_2099_4999,2026-08-20,2026-08-20,2,101.57,2025Q4,PartyBundle
4,6980B3A8474F4C40,2026-08-19,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,99.99,daily_offers_5099_plus,2026-08-19,2026-08-19,1,99.99,2025Q4,PartyBundle
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8574,106B0E0ED94D9F,2026-08-18,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-18,2026-08-18,1,0.42,2025Q4,CafeBundle
8575,9CD41EBD47773AFE,2026-08-24,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-24,2026-08-24,1,0.42,2025Q4,CafeBundle
8576,468E1C80CBE8CFB8,2026-08-30,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-30,2026-08-30,1,0.42,2025Q4,CafeBundle
8577,D5BEB8FD4DBCD610,2026-08-31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,RotatingIncPack,RotatingOffersIncPack,0.99,daily_offers_0099_0499,2026-08-31,2026-08-31,1,0.42,2025Q4,CafeBundle


In [14]:
daily_by_quarter = data.groupby(['dt', 'offer_quarter','price_point_usd'], as_index=False).agg(
    usd_revenue=('usd_revenue', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_usd_revenue=('usd_revenue', 'mean'),
).sort_values('dt')

daily_by_quarter['combined_dim'] = '$' + daily_by_quarter['price_point_usd'].astype(str) + ' · ' + daily_by_quarter['offer_quarter']

daily_by_quarter.sort_values(by=['dt', 'combined_dim'], inplace=True)

quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

fig = px.bar(
    daily_by_quarter, 
    x='dt', 
    y='usd_revenue', 
    color='combined_dim',
    #color_discrete_map=quarter_colors,
    custom_data=['n_trans','offer_quarter','price_point_usd'],
)
fig.update_traces(
    #mode='lines+markers',
    hovertemplate='Date: %{x|%Y-%m-%d}<br>Revenue: $%{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<br>Offer Quarter: %{customdata[1]}<br>Price Point: $%{customdata[2]:,.2f}<extra></extra>',
)
fig.update_layout(
    title='Daily Revenue by Offer Quarter and Price Point',
    xaxis=dict(title='Date'),
    yaxis=dict(title='Revenue (USD)', tickprefix='$'),
    height=450, 
    width=1100,
    legend=dict(orientation='v', yanchor='top', xanchor='right',title=None),
)
fig.show()

## Top Offers by Revenue per Active Day

Normalized ranking — controls for offers that have only been live a few days (e.g. new 2026Q3 variants) vs. offers that have run the full window. Restricted to offers with >= 2 active days, since single-day figures are too noisy to rank on.

In [15]:
top_per_day = product_summary[product_summary['active_days'] >= 2].nlargest(top_n, 'revenue_per_day').sort_values('revenue_per_day')
top_per_day = top_per_day.assign(
    label=top_per_day['bundle_type'] + ' · ' + top_per_day['offer_quarter'] + ' · $' + top_per_day['price_point_usd'].astype(str)
)

fig = px.bar(
    top_per_day, x='revenue_per_day', y='label', orientation='h',
    custom_data=['usd_revenue', 'active_days', 'price_point_usd'],
)
fig.update_traces(
    marker_color='steelblue',
    hovertemplate=(
        'Revenue/day: $%{x:,.0f}<br>Total revenue: $%{customdata[0]:,.0f}<br>'
        'Active days: %{customdata[1]}<br>Price: $%{customdata[2]:.2f}<extra></extra>'
    ),
)
fig.update_layout(
    title=f'Top {top_n} Daily Offers by Revenue per Active Day',
    xaxis=dict(title='Revenue per Active Day (USD)', tickprefix='$'),
    yaxis=dict(title=None, automargin=True),
    height=550, width=1100,
    margin=dict(l=10),
)
fig.show()

## Checkout Conversion: Started vs Completed

`fact_dt_user_product_iap_revenue` only sees completed purchases. `rp_IAPStart` fires when a player taps "buy" on a specific `product_id`, before success/fail/cancel is known — it's not a pure "offer shown" signal (no impression-level event exists for `daily_offers` in dbt today), but it does let us see checkout attempts that didn't convert, which the revenue-only charts above can't.

In [16]:
starts_query_location = './sql/daily_offers_starts.sql'
starts_query_name = 'daily_offers_starts'

starts_cost_info = bqc.print_cost_estimate(query=starts_query_location, is_path=True)

This query will process 3.7 MB when run.
Estimated query cost: $0.00


In [17]:
refresh_starts_data = True

starts = pd.DataFrame()

if refresh_starts_data:
    starts = bqc.get(query=starts_query_location, is_path=True)
    starts.to_pickle(f'./data/{starts_query_name}.pkl')
else:
    starts = pd.read_pickle(f'./data/{starts_query_name}.pkl')

In [18]:
starts_by_product = starts.groupby('product_id', as_index=False)['n_starts'].sum()

conversion = product_summary[[
    'product_id', 'offer_quarter', 'bundle_type', 'product_type',
    'product_price_group', 'price_point_usd', 'n_trans', 'usd_revenue',
]].merge(starts_by_product, on='product_id', how='left')
conversion['n_starts'] = conversion['n_starts'].fillna(0)
conversion['conversion_rate_pct'] = (conversion['n_trans'] / conversion['n_starts'].mask(conversion['n_starts'] == 0) * 100).round(1)

conversion.sort_values('n_starts', ascending=False)

,product_id,offer_quarter,bundle_type,product_type,product_price_group,price_point_usd,n_trans,usd_revenue,n_starts,conversion_rate_pct
31,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,GardenBundle,RotatingIncPack,daily_offers_0599_0999,9.99,1246,13463.49,1430,87.1
32,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,GardenBundle,RotatingIncPack,daily_offers_1099_1999,12.99,630,8912.07,720,87.5
33,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,GardenBundle,RotatingIncPack,daily_offers_1099_1999,14.99,589,9419.45,679,86.7
35,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,GardenBundle,RotatingIncPack,daily_offers_2099_4999,24.99,556,14711.43,648,85.8
27,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,CafeBundle,RotatingIncPack,daily_offers_0599_0999,7.99,518,4484.85,615,84.2
30,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,CafeBundle,RotatingIncPack,daily_offers_0599_0999,6.99,497,3789.80,602,82.6
28,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,CafeBundle,RotatingIncPack,daily_offers_0099_0499,3.99,481,2090.91,567,84.8
47,TimedAlbum-DailyOffers-2026Q3-RotatingOffersIn...,2026Q3,GardenBundle,RotatingIncPack,daily_offers_1099_1999,14.99,451,7338.80,535,84.3
34,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,GardenBundle,RotatingIncPack,daily_offers_1099_1999,19.99,441,9399.22,515,85.6
29,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,CafeBundle,RotatingIncPack,daily_offers_0599_0999,5.99,416,2758.04,484,86.0


### Overall, by Offer Quarter

In [19]:
conv_by_quarter = conversion.groupby('offer_quarter', as_index=False).agg(
    n_starts=('n_starts', 'sum'),
    n_trans=('n_trans', 'sum'),
)
conv_by_quarter['conversion_rate_pct'] = (conv_by_quarter['n_trans'] / conv_by_quarter['n_starts'] * 100).round(1)

conv_by_quarter_long = conv_by_quarter.melt(
    id_vars=['offer_quarter', 'conversion_rate_pct'],
    value_vars=['n_starts', 'n_trans'],
    var_name='metric', value_name='count',
)
conv_by_quarter_long['metric'] = conv_by_quarter_long['metric'].map({'n_starts': 'Started', 'n_trans': 'Completed'})
conv_by_quarter_long['label_text'] = conv_by_quarter_long.apply(
    lambda r: f"{r['conversion_rate_pct']:.1f}% conv." if r['metric'] == 'Completed' else '', axis=1
)

fig = px.bar(
    conv_by_quarter_long, x='offer_quarter', y='count', color='metric', barmode='group',
    color_discrete_map={'Started': '#2a78d6', 'Completed': '#1baf7a'},
    text='label_text',
)
fig.update_traces(textposition='outside', hovertemplate='%{y:,.0f}<extra></extra>')
fig.update_layout(
    title='Checkout Starts vs Completed Purchases by Offer Quarter',
    xaxis=dict(title=None, type='category'),
    yaxis=dict(title='Count'),
    height=450, width=800,
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5, title=None),
)
fig.show()

### By Price Tier and Offer Quarter

Where the overall number could hide it — do higher-priced tiers convert worse under 2026Q3 than they did under 2025Q4? Restricted to tier/quarter combinations with at least one recorded checkout start.

In [20]:
price_tier_order = [
    'daily_offers_0099_0499', 'daily_offers_0599_0999', 'daily_offers_1099_1999',
    'daily_offers_2099_4999', 'daily_offers_5099_plus',
]

conv_by_tier = conversion.groupby(['offer_quarter', 'product_price_group'], as_index=False).agg(
    n_starts=('n_starts', 'sum'),
    n_trans=('n_trans', 'sum'),
)
conv_by_tier = conv_by_tier[conv_by_tier['n_starts'] > 0]
conv_by_tier['conversion_rate_pct'] = (conv_by_tier['n_trans'] / conv_by_tier['n_starts'] * 100).round(1)
conv_by_tier['price_tier'] = conv_by_tier['product_price_group'].str.replace('daily_offers_', '', regex=False)

tier_order_labels = [t.replace('daily_offers_', '') for t in price_tier_order]
quarter_colors = {'2025Q4': '#2a78d6', '2026Q3': '#eb6834'}

fig = px.bar(
    conv_by_tier, x='price_tier', y='conversion_rate_pct', color='offer_quarter',
    barmode='group',
    color_discrete_map=quarter_colors,
    category_orders={'price_tier': tier_order_labels},
    custom_data=['n_starts', 'n_trans'],
)
fig.update_traces(
    hovertemplate='Conversion: %{y:.1f}%<br>Started: %{customdata[0]:,.0f}<br>Completed: %{customdata[1]:,.0f}<extra></extra>'
)
fig.update_layout(
    title='Checkout Conversion by Price Tier and Offer Quarter',
    xaxis=dict(title='Price Tier (USD)'),
    yaxis=dict(title='Conversion Rate', ticksuffix='%'),
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

## Player Price Migration: Did Players Move to a Higher Price?

Comparing "same bundle, same price" across quarters mixes different days and different player populations — not a fair test. This instead tracks individual `RotatingIncPack` players: for each player who bought a given `bundle_type` both before and on/after the 2026Q3 launch (2026-08-26), compare their own last pre-launch price against their own first post-launch price. Same player, so the population is controlled by construction — only time changes.

In [21]:
launch_date = pd.Timestamp('2026-08-26').date()
migration_base = data[data['product_type'] == 'RotatingIncPack']

before_launch = migration_base[migration_base['dt'] < launch_date].sort_values('dt')
after_launch = migration_base[migration_base['dt'] >= launch_date].sort_values('dt')

baseline = before_launch.groupby(['user_id', 'bundle_type'], as_index=False)['price_point_usd'].last().rename(columns={'price_point_usd': 'baseline_price'})
post_first = after_launch.groupby(['user_id', 'bundle_type'], as_index=False)['price_point_usd'].first().rename(columns={'price_point_usd': 'post_price'})

migration = baseline.merge(post_first, on=['user_id', 'bundle_type'], how='inner')
migration['direction'] = pd.cut(
    migration['post_price'] - migration['baseline_price'],
    bins=[-float('inf'), -0.001, 0.001, float('inf')],
    labels=['Moved lower', 'Stuck same', 'Moved higher'],
)

migration

,user_id,bundle_type,baseline_price,post_price,direction
0,10677E93CD2AC270,GardenBundle,14.99,14.99,Stuck same
1,1080F1B06153C00B,CafeBundle,7.99,7.99,Stuck same
2,10C7A23E5DA34D43,CafeBundle,4.99,4.99,Stuck same
3,10E1BA15E26F8385,CafeBundle,6.99,4.99,Moved lower
4,11AC4098836B24C0,GardenBundle,24.99,24.99,Stuck same
...,...,...,...,...,...
1025,FF5F5CEE6078A135,GardenBundle,19.99,14.99,Moved lower
1026,FF74A4C8FB83F576,CafeBundle,2.99,2.99,Stuck same
1027,FF7F6363A46125CC,CafeBundle,6.99,5.99,Moved lower
1028,FF83CEC5F6C1D45B,GardenBundle,9.99,14.99,Moved higher


In [22]:
migration_by_bundle = migration.groupby(['bundle_type', 'direction'], observed=True).size().reset_index(name='n_players')
migration_by_bundle['pct'] = migration_by_bundle.groupby('bundle_type')['n_players'].transform(lambda s: (s / s.sum() * 100).round(1))

direction_colors = {'Moved lower': '#1baf7a', 'Stuck same': '#2a78d6', 'Moved higher': '#eb6834'}

fig = px.bar(
    migration_by_bundle, x='bundle_type', y='pct', color='direction',
    color_discrete_map=direction_colors,
    category_orders={'direction': ['Moved lower', 'Stuck same', 'Moved higher']},
    custom_data=['n_players'],
)
fig.update_traces(hovertemplate='%{y:.1f}%<br>Players: %{customdata[0]:,.0f}<extra></extra>')
fig.update_layout(
    barmode='stack',
    title='Player Price Migration by Bundle Type (Before → After 2026Q3 Launch)',
    xaxis=dict(title=None),
    yaxis=dict(title='Share of Players', ticksuffix='%'),
    height=450, width=800,
    legend=dict(orientation='h', yanchor='bottom', y=-0.25, xanchor='center', x=0.5, title=None),
)
fig.show()

## Offer Value: Rewards per Dollar

2026Q3 was designed to move players to a higher price tier while also giving more rewards — the fair test of that isn't "same price, different quarter" (the previous section), it's "does the reward you get keep pace with what you pay." `rpp_lighthouse_products` (the IAP catalogue, joined by `product_id`) carries `product_contents_array` and `product_bonus_array` — item/currency grants per product, as `(id, cnt)` pairs. `total_value` sums every `cnt` across both arrays (a blunt unweighted count — gems and energy dominate it, cosmetic items barely register), and `value_per_dollar = total_value / price` is the resulting "how much stuff per dollar" metric.

In [23]:
rewards_query_location = './sql/daily_offers_rewards.sql'
rewards_query_name = 'daily_offers_rewards'

rewards_cost_info = bqc.print_cost_estimate(query=rewards_query_location, is_path=True)

This query will process 29.5 MB when run.
Estimated query cost: $0.00


In [24]:
refresh_rewards_data = True

rewards = pd.DataFrame()

if refresh_rewards_data:
    rewards = bqc.get(query=rewards_query_location, is_path=True)
    rewards.to_pickle(f'./data/{rewards_query_name}.pkl')
else:
    rewards = pd.read_pickle(f'./data/{rewards_query_name}.pkl')

In [25]:
rewards['total_value'] = rewards['total_contents_cnt'].fillna(0) + rewards['total_bonus_cnt'].fillna(0)

value = product_summary[[
    'product_id', 'offer_quarter', 'bundle_type', 'product_type', 'product_price_group', 'price_point_usd', 'n_trans',
]].merge(rewards[['product_id', 'total_value']], on='product_id', how='left')
value['value_per_dollar'] = (value['total_value'] / value['price_point_usd']).round(1)

value.sort_values(['bundle_type', 'price_point_usd'])

,product_id,offer_quarter,bundle_type,product_type,product_price_group,price_point_usd,n_trans,total_value,value_per_dollar
0,TimedAlbum-DailyOffers-2025Q4-NonPayer-BundleL...,2025Q4,BundleLarge,NonPayer,daily_offers_0599_0999,9.99,10,903,90.4
3,TimedAlbum-DailyOffers-2025Q4-NonPayerIncPack-...,2025Q4,BundleLarge,NonPayerIncPack,daily_offers_0599_0999,9.99,7,904,90.5
1,TimedAlbum-DailyOffers-2025Q4-NonPayer-BundleM...,2025Q4,BundleMedium,NonPayer,daily_offers_0099_0499,4.99,19,296,59.3
4,TimedAlbum-DailyOffers-2025Q4-NonPayerIncPack-...,2025Q4,BundleMedium,NonPayerIncPack,daily_offers_0099_0499,4.99,2,297,59.5
2,TimedAlbum-DailyOffers-2025Q4-NonPayer-BundleS...,2025Q4,BundleSmall,NonPayer,daily_offers_0099_0499,2.99,52,228,76.3
5,TimedAlbum-DailyOffers-2025Q4-NonPayerIncPack-...,2025Q4,BundleSmall,NonPayerIncPack,daily_offers_0099_0499,2.99,35,229,76.6
6,TimedAlbum-DailyOffers-2025Q4-RotatingOffers-C...,2025Q4,CafeBundle,Rotating,daily_offers_0099_0499,0.99,30,160,161.6
23,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,CafeBundle,RotatingIncPack,daily_offers_0099_0499,0.99,329,160,161.6
7,TimedAlbum-DailyOffers-2025Q4-RotatingOffers-C...,2025Q4,CafeBundle,Rotating,daily_offers_0099_0499,1.99,38,196,98.5
24,TimedAlbum-DailyOffers-2025Q4-RotatingOffersIn...,2025Q4,CafeBundle,RotatingIncPack,daily_offers_0099_0499,1.99,305,196,98.5


In [26]:
value[[
    'bundle_type', 'product_type', 'price_point_usd', 'offer_quarter', 'n_trans', 'total_value', 'value_per_dollar',
]].sort_values(['bundle_type', 'product_type', 'price_point_usd', 'offer_quarter'])

,bundle_type,product_type,price_point_usd,offer_quarter,n_trans,total_value,value_per_dollar
0,BundleLarge,NonPayer,9.99,2025Q4,10,903,90.4
3,BundleLarge,NonPayerIncPack,9.99,2025Q4,7,904,90.5
1,BundleMedium,NonPayer,4.99,2025Q4,19,296,59.3
4,BundleMedium,NonPayerIncPack,4.99,2025Q4,2,297,59.5
2,BundleSmall,NonPayer,2.99,2025Q4,52,228,76.3
5,BundleSmall,NonPayerIncPack,2.99,2025Q4,35,229,76.6
6,CafeBundle,Rotating,0.99,2025Q4,30,160,161.6
7,CafeBundle,Rotating,1.99,2025Q4,38,196,98.5
8,CafeBundle,Rotating,2.99,2025Q4,28,228,76.3
40,CafeBundle,Rotating,2.99,2026Q3,1,228,76.3


### Value per Dollar by Price Tier and Offer Quarter

Same price-tier cut as the conversion chart above, so the two can be read side by side: did the tiers where 2026Q3 converted worse also deliver worse value per dollar?

In [27]:
value['weighted'] = value['value_per_dollar'] * value['n_trans']
value_by_tier = value.groupby(['offer_quarter', 'product_price_group'], as_index=False).agg(
    weighted_sum=('weighted', 'sum'),
    n_trans=('n_trans', 'sum'),
    avg_price_point_usd=('price_point_usd', 'mean')
)
value_by_tier['value_per_dollar_avg'] = (value_by_tier['weighted_sum'] / value_by_tier['n_trans']).round(1)
value_by_tier['price_tier'] = value_by_tier['product_price_group'].str.replace('daily_offers_', '', regex=False)
value_by_tier = value_by_tier[value_by_tier['n_trans'] > 0]

fig = px.bar(
    value_by_tier, x='price_tier', y='value_per_dollar_avg', color='offer_quarter',
    barmode='group',
    color_discrete_map=quarter_colors,
    category_orders={'price_tier': tier_order_labels},
    custom_data=['n_trans'],
)
fig.update_traces(
    hovertemplate='Value/$: %{y:,.0f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>'
)
fig.update_layout(
    title='Purchase-Weighted Value per Dollar by Price Tier and Offer Quarter',
    xaxis=dict(title='Price Tier (USD)'),
    yaxis=dict(title='Value per Dollar (reward units)'),
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

# Average price_point_usd by price tier and offer quarter

fig = px.bar(
    value_by_tier, x='price_tier', y='avg_price_point_usd', color='offer_quarter',
    barmode='group',
    color_discrete_map=quarter_colors,
    category_orders={'price_tier': tier_order_labels},
    custom_data=['n_trans'],
)
fig.update_traces(
    hovertemplate='Avg. Price Point (USD): %{y:,.2f}<br>Transactions: %{customdata[0]:,.0f}<extra></extra>'
)
fig.update_layout(
    title='Average Price Point by Price Tier and Offer Quarter',
    xaxis=dict(title='Price Tier (USD)'),
    yaxis=dict(title='Average Price Point (USD)'),
    height=450, width=1000,
    legend=dict(orientation='h', yanchor='bottom', y=-0.3, xanchor='center', x=0.5, title=None),
)
fig.show()

In [28]:
export_notebook_html(
# Exports the notebook to HTML for sharing and archival
    notebook_path='./daily_offers.ipynb',
    output_path='./daily_offers.html',
)

Saved to daily_offers.html


PosixPath('daily_offers.html')